# 11_gain — 효용 U 대신 첫 시도 이득 G로 다시 평가 (EV-10)

[최종 실험 계획서](../../../docs/ml/ROUTER_FINAL_EXPERIMENT_PLAN.md)의 라우터 비교를 **학습 없이** 다시 평가한다. 04~07a가 저장한 모든 설정의 점수를 그대로 쓴다.

$$G = A - H(C)$$

- **A:** 라우터가 고른 시작 단계의 첫 시도 성공(집단 풀 확률의 평균)
- **C:** 첫 시도 비용의 평균(Claude 1:3:5, .2/.6/1.0)
- **H(C):** 첫 시도 비용이 C 이하인 고정 시작 단계(혼합 포함)가 낼 수 있는 가장 좋은 첫 시도 성공. 07b의 envelope에서 κ = 0, 첫 시도만 본 것이다. 비싼 단계가 오히려 못하면 기준선은 내려가지 않는다.
- λ(비용–해결률 환율)와 κ(실패 가중치)가 G에 들어가지 않는다. 상향(escalation)은 G에 없다.

**두 단계.**

1. **gain lock (dev만):** 모든 설정을 07b와 똑같이 보정(같은 isotonic 어댑터, 같은 5분할 교차 보정)하고, 방법마다 **τ = .8 규칙의 dev G(원천 4개 평균)**가 가장 큰 설정을 고른다. 한 번에 고르기 규칙 `argmax(p̂ − λ × 가격)`은 원천·비용 수준마다 dev 첫 시도 비용이 고정 시작 범위의 25·50·75%(1:3:5에서 .4/.6/.8)에 가장 가까운 λ를 고른다. **여기서 멈추고 lock 요약을 보고한다.**
2. **test:** gain lock을 그대로 써서 test를 연다. 아무것도 고르지 않는다. 질문 묶음 단위 paired bootstrap 10,000회(08과 같은 표본).

**출력.** 원천별·범위별(원천 4개 평균, A 분할, 외부) G와 95% 구간, 비용 민감도(집단 가격·길이 반영, τ = .8), 라우터 쌍의 차이, 풀리는 정도별 층(기술 통계), H5 seed 3개(기술 통계, τ = .8). 운영 R0와 RT-08은 07b 보정을 그대로 쓰는 참고 행이고 순위에 넣지 않는다. `needed_tier`(정답 풀 확률로 τ = .8, 참값을 아는 기준)도 참고로 싣는다.

**런타임: CPU.** gain lock 10~20분(설정 45개 보정), 리허설 몇 분, test 20~30분. 입력은 08과 같다. 소스는 Drive `code/`의 09 소스 묶음(`router-final-source-seeds.zip`)에 새 파일 `gain_eval.py` 하나를 더해 쓰고, 둘 다 SHA-256으로 확인한다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import hashlib, json, shutil, subprocess, sys, zipfile
RUN = Path('/content/drive/MyDrive/LLM_Router/router-final/runs/run001')
CODE = RUN / 'code'
sha = lambda path: hashlib.sha256(Path(path).read_bytes()).hexdigest()
# 기반 소스: 09_seeds 소스 묶음(라우터 최종 실험 코드의 마지막 수정 뒤에 만든 것)
BASE_ZIP = CODE / 'router-final-source-seeds.zip'
assert sha(BASE_ZIP) == BASE_ZIP.with_suffix('.sha256').read_text().split()[0], '기반 소스 ZIP이 바뀌었다'
REPO = Path('/content/router-final-source')
with zipfile.ZipFile(BASE_ZIP) as z:
    for name in z.namelist():
        assert (REPO / name).resolve().is_relative_to(REPO.resolve()), name
    z.extractall(REPO)
bundle = json.loads((REPO / 'source-bundle-manifest.json').read_text())
for name, digest in bundle.items():
    assert sha(REPO / name) == digest, name
# gain_eval.py가 그대로 불러 쓰는 모듈은 기반 묶음과 저장소(main)의 판이 같아야 한다
SAME = {
    'lock.py': '04197717a5659f6e294fcd530917d5a49b7c07be40212856e9c8e61f0c234232',
    'test_eval.py': 'c5dc3451abc9bc2058f70f2b3bf0e80a854d4e8dec707622434ea1dba4a8391f',
    'common_track.py': 'fad9e50679a521d950d7162f249e6df2d5afa62925f7ce3d595d875de700360c',
    'categories.py': 'eb2cea4cb798fa84a3c5eec2195ed7ef7a449506390c351bc39c02152fe8a01b',
    'prepare.py': '2dd3fcbfa57f4a7d49a2156ebfa3c23a07438fd19a35eded92a178169e125054',
    'config.json': '43c9087e6bf2e8446b41ed867a0a296ad89f77a0322f61011109d70fe2138562',
}
for name, digest in SAME.items():
    assert bundle[f'ml/notebooks/router_final/{name}'] == digest, f'{name}: 기반 묶음의 판이 다르다'
# 새 파일 하나만 따로 올렸다
GAIN = CODE / 'gain_eval.py'
assert sha(GAIN) == 'eaada245d737ee9d44588f2381ad2790413688a21d6d49342d43d01220bb5330', 'gain_eval.py가 바뀌었다'
shutil.copy(GAIN, REPO / 'ml/notebooks/router_final/gain_eval.py')
DATA_ROOT = RUN / 'inputs' / 'test-data'
for name, digest in json.loads((DATA_ROOT / 'test-data-manifest.json').read_text())['files'].items():
    assert sha(DATA_ROOT / name) == digest, name
assert sys.version_info[:2] in {(3, 11), (3, 12)}, '런타임 버전을 2026.07(Python 3.12)로 바꾸세요'
subprocess.run([sys.executable, '-m', 'pip', 'install', 'scikit-learn==1.6.1', 'numpy==2.1.2', 'pandas==2.3.3', 'pyarrow==21.0.0'], check=True)

설치가 끝나면 아래 셀부터 이어서 실행한다. 계산은 새 프로세스(`subprocess`)에서 하므로 설치한 버전이 바로 쓰이고, 세션을 다시 시작하지 않아도 된다.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys
RUN = Path('/content/drive/MyDrive/LLM_Router/router-final/runs/run001')
REPO = Path('/content/router-final-source')
DATA_ROOT = RUN / 'inputs' / 'test-data'

def run_logged(*args):
    """Run gain_eval.py and stream its log into this cell (subprocess.run hides it in Colab)."""
    proc = subprocess.Popen([sys.executable, str(REPO / 'ml/notebooks/router_final/gain_eval.py'), *map(str, args)],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                            env=dict(os.environ, PYTHONUNBUFFERED='1'))
    for line in proc.stdout:
        print(line, end='')
    if proc.wait():
        raise RuntimeError(f'실패(exit {proc.returncode}). 위 로그의 마지막 부분(Traceback)을 보내 주세요')

## 1. gain lock (dev만, test를 읽지 않음)

모든 설정을 다시 보정하고 G 기준으로 설정을 고른다. 07b와 같은 설정을 고른 방법은 07b 보정을 정확히 재현하는지 확인하고(차이 > 1e-9면 멈춤), 바뀐 설정은 요약에 표시한다. **끝나면 `gain-lock-summary.md`를 보고하고 확인을 받은 뒤 2로 간다.**

In [ ]:
GAIN_LOCK = RUN / 'gain-lock'
assert not GAIN_LOCK.exists(), '이미 gain lock이 있다. 다시 만들려면 폴더를 다른 이름으로 옮긴 뒤 실행'
run_logged('lock', '--run-dir', RUN, '--repo', REPO, '--output-dir', GAIN_LOCK)
print((GAIN_LOCK / 'gain-lock-summary.md').read_text())

## 2. 리허설 (dev, test를 읽지 않음)

같은 코드를 dev 결과로 돌려 경로와 환경을 확인한다. bootstrap 1,000회. 결과는 Colab 디스크에만 쓰고, dev에서 맞춘 보정을 dev에 다시 쓴 값이라 낙관적이므로 해석하지 않는다.

In [ ]:
import shutil
REHEARSAL = Path('/content/rehearse-gain')
shutil.rmtree(REHEARSAL, ignore_errors=True)  # 리허설 결과는 버리는 값이라 다시 할 때 지운다
run_logged('test', '--run-dir', RUN, '--repo', REPO, '--data-root', DATA_ROOT, '--output-dir', REHEARSAL,
           '--rehearse-on-dev', '--iterations', 1000)
print((REHEARSAL / 'gain-summary.md').read_text()[:2000])

## 3. test (gain lock 확인 뒤, 한 번만)

bootstrap 10,000회. 출력 폴더 `gain-test/`가 이미 있으면 멈춘다.

In [ ]:
GAIN_TEST = RUN / 'gain-test'
assert not GAIN_TEST.exists(), '이미 열었다. 실패한 실행을 다시 할 때만 폴더를 다른 이름으로 옮긴 뒤 실행'
run_logged('test', '--run-dir', RUN, '--repo', REPO, '--data-root', DATA_ROOT, '--output-dir', GAIN_TEST)

In [ ]:
from pathlib import Path
import json
GAIN_TEST = Path('/content/drive/MyDrive/LLM_Router/router-final/runs/run001/gain-test')
print((GAIN_TEST / 'gain-summary.md').read_text())
manifest = json.loads((GAIN_TEST / 'manifest.json').read_text())
print(json.dumps({k: manifest[k] for k in ('counts', 'seconds', 'outputs')}, indent=1))

## 다음

`gain-test/`의 `gain-summary.md`를 보고한다. U 기준 결과(08)는 그대로 두고, G는 "첫 시도에서 단계를 잘 고르는가"를 보는 두 번째 관점으로 함께 쓴다. 가망 없는 문항에서는 G와 U가 반대 행동을 보상하므로(G는 싼 단계, U는 비싼 단계) 층별 표를 함께 본다.